**Búsqueda Semántica con Embeddings de GPT-4**

Este programa permite realizar búsqueda semántica utilizando el embedding de una consulta de un usuario y entregando los textos más similares de un dataset en base a medidas de similitud coseno.

Los embeddings se obtienen desde múltiples modelos de embeddings de GPT3-5 disponibles desde OpenAI (i.e., *text-embedding-ada-002*).

Primero, instalamos algunos paquetes para acceder a los modelos:

In [ ]:
!pip install openai

Importamos algunas librerías y métodos desde OpenAI para obtener embeddings y calcular similitud coseno entre vectores:

In [3]:
import pandas as pd
import numpy as np
import pprint
from openai import OpenAI
from numpy import dot # Import dot product for cosine similarity
from numpy.linalg import norm # Import norm for cosine similarity

#from openai.embeddings_utils import get_embedding, cosine_similarity

In [11]:
# Open AI API-key
from google.colab import files
from IPython.display import clear_output

files.upload() # subir archivo con apikey de openai propio
clear_output() # no muestra contenido del apikey

In [12]:
def get_api_key():
    with open('idsa_openai_key.txt', 'r') as fp: #acá reemplazar x el nombre de tu archivo
        key = fp.read()
    return key

# Enter your OpenAI API key here
openai_api_key = get_api_key()

In [13]:
!wget https://github.com/palasatenea66/DATASETS/raw/main/sentiments-ingles.csv # descarga archivo con datos

--2025-05-23 01:05:40--  https://github.com/palasatenea66/DATASETS/raw/main/sentiments-ingles.csv
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/palasatenea66/DATASETS/main/sentiments-ingles.csv [following]
--2025-05-23 01:05:40--  https://raw.githubusercontent.com/palasatenea66/DATASETS/main/sentiments-ingles.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 163997 (160K) [text/plain]
Saving to: ‘sentiments-ingles.csv.1’

sentiments-ingles.c 100%[===================>] 160.15K  --.-KB/s    in 0.01s   

2025-05-23 01:05:41 (13.8 MB/s) - ‘sentiments-ingles.csv.1’ saved [163997/163997]



Definimos la función **BuscarComentarios(Opiniones,Consulta,n)** que busca las similitudes (coseno) entre una **consulta** y las **Opiniones**, y entrega los **n** meuores resultados. Para esto, la consulta se convierte a su correspondiente embedding:

In [14]:
# Define a function for cosine similarity using numpy
def cosine_similarity(a, b):
    return dot(a, b)/(norm(a)*norm(b))

# Definimos la función BuscarComentarios(Opiniones,Consulta,n) que busca las similitudes (coseno) entre una consulta y las Opiniones, y entrega los n meuores resultados. Para esto, la consulta se convierte a su correspondiente embedding:
def BuscarComentarios(Opiniones, consulta, n=3):
    # Initialize the OpenAI client
    client = OpenAI(api_key=openai_api_key) # Use the API key set globally

    # Obtener embedding de la consulta using the new client method
    response = client.embeddings.create(
        model=MODELO_EMBEDDING,
        input=consulta
    )
    EmbeddingConsulta = response.data[0].embedding

    # Calcular la similitud coseno entre el embedding de la consulta
    # y el embedding de cada una de las opiniones
    # Cada valor de similitud coseno se almacena en una nueva columna "similitud"
    # de la lista de opiniones
    # Ensure the embeddings from the DataFrame are in a format compatible with numpy (e.g., list or array)
    Opiniones["similitud"] = Opiniones.embedding.apply(lambda x: cosine_similarity(np.array(x), np.array(EmbeddingConsulta)))
    # Ordenar resultados en forma descendente
    results = Opiniones.sort_values("similitud", ascending=False).head(n)
    return results

In [15]:
# Choose one of the embedding models to use
MODELO_EMBEDDING = "text-embedding-ada-002"

# Cargar comentarios desde archivo "sentiments-ingles.csv"
# Assuming the first column (index 0) contains the text
Opiniones = pd.read_csv('/content/sentiments-ingles.csv', header=None)

# Create a new column "embedding" with the embeddings of each of the comments
# Initialize the OpenAI client again for getting embeddings for the dataframe
client = OpenAI(api_key=openai_api_key)


In [16]:
# Define a helper function to get embeddings for a text
def get_embedding_for_text(text, model):
    response = client.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding

Opiniones["embedding"] = Opiniones[0].apply(lambda x: get_embedding_for_text(x, MODELO_EMBEDDING))

In [17]:
results = BuscarComentarios(Opiniones, "borderlands quite", 5)
print(results[0])

872         Borderlands stuff is quite fun
870       Borderlands 3 is quite different
867           Borderlands 3 is quite fun  
871    Borderlands 3 is actually quite fun
868            Borderlands 3 is really fun
Name: 0, dtype: object
